# Notebook 13 (Revised) — Selection-lock and final-refit **preflight only**

**Status:** Completed: executed and verified read-only final-refit preflight on the local SOLETE dataset. No model refitting or terminal evaluation performed. This notebook follows frozen revised Notebooks 10–12 and the selection lock committed as **`c6a9162`**. It checks the **already-selected** final LightGBM candidates, the locked code and package environment, the **timestamp-only** refit masks, and the development-only F1/F2 model inputs. **It cannot fit models or calculate forecasts**: there is deliberately **no `fit()` or `predict()` call**.

| Horizon | Frozen primary candidate | Feature set | Refit row count required |
|:--|:--|:--|--:|
| 15 minutes | LightGBM C1 | F1 (118 F0 + 15 valid-time Gregorian) | 104,817 |
| 30 minutes | LightGBM C1 | F2 (F1 + 5 valid-time solar) | 104,814 |
| 60 minutes | LightGBM C1 | F2 | 104,808 |

The historical June–August 2019 period is **not independently untouched in the broader project**. It was examined by historical notebooks. This stage does not use any terminal *measurement* or *target*, and does not authorize later scoring.

**Run:** From the repository, open this notebook with the existing `.venv` kernel, restart and **Run All**. It reads HDF5 timestamps for the full study solely to establish boundaries; **measurement values are read only for timestamps strictly earlier than `2019-05-31 23:00 UTC`**. F2 solar computations may take some time on the CPU. No outputs besides notebook display are explicitly written.

## 1. Read-only execution guard and repository discovery

Safety switches are assertions, **not enable flags**. A future refit must be implemented separately after explicit approval; turning a flag here cannot launch one. A failed invariant stops the preflight without attempting an alternative training or evaluation path.

In [1]:
from __future__ import annotations
import hashlib
import importlib.metadata as mdist
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# No future model training or terminal scoring can be enabled in this notebook.
REFIT_AUTHORIZED = False
TERMINAL_EVALUATION_AUTHORIZED = False
assert REFIT_AUTHORIZED is False and TERMINAL_EVALUATION_AUTHORIZED is False

ROOT_CANDIDATES = []
if os.environ.get("SOLETE_PROJECT_ROOT"):
    ROOT_CANDIDATES.append(Path(os.environ["SOLETE_PROJECT_ROOT"]).expanduser())
ROOT_CANDIDATES.extend([Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next(
    (p.resolve() for p in ROOT_CANDIDATES
     if (p / "scripts/verify_revised_selection_lock.py").is_file()
     and (p / "results/development_only/revised_model_selection_lock_v1.json").is_file()
     and (p / ".git").exists()), None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run inside the verified SOLETE checkout (or set SOLETE_PROJECT_ROOT)")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SELECTION_COMMIT = "c6a9162"
LOCK_REL = "results/development_only/revised_model_selection_lock_v1.json"
LOCK_PATH = PROJECT_ROOT / LOCK_REL
lock_bytes = LOCK_PATH.read_bytes()
lock = json.loads(lock_bytes.decode("utf-8"))
assert lock["archived_audit"]["archived_fold_scores"] == 1665
assert lock["evaluation_policy"]["extra_training_authorized"] is False
assert lock["evaluation_policy"]["terminal_access_authorized"] is False
assert lock["refit_protocol"]["do_not_fit_or_score_terminal_data"] is True
print("Notebook 13 read-only guard: PASSED")
print("Repository:", PROJECT_ROOT)
print("Frozen selection source:", SELECTION_COMMIT)
print("Selection manifest SHA-256:", hashlib.sha256(lock_bytes).hexdigest())

Notebook 13 read-only guard: PASSED
Repository: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project
Frozen selection source: c6a9162
Selection manifest SHA-256: 6c61307b0048e00dc54e232fef52874cf36336e8539f650dd3376fb523473d4f


## 2. Confirm the lock is the **exact committed artifact**

The original JSON still contains `freeze_state=PROPOSED_LOCK_NOT_YET_COMMITTED` because that was its preparation-time label. **Do not edit it now** to change that text: the effective freeze event is Git commit `c6a9162`. We therefore independently require byte-for-byte identity with the lock **as committed** and run the already-tested, read-only verifier. This is stronger than relying on an editable status string.

In [2]:
def git(*args, check=True, text=True):
    return subprocess.run(
        ["git", *args], cwd=PROJECT_ROOT, capture_output=True,
        check=check, text=text,
    )

assert git("merge-base", "--is-ancestor", SELECTION_COMMIT, "HEAD").returncode == 0
committed_lock = git("show", f"{SELECTION_COMMIT}:{LOCK_REL}", text=False).stdout
assert lock_bytes == committed_lock, "The selection lock differs from the frozen Git commit"
assert lock["freeze_state"] == "PROPOSED_LOCK_NOT_YET_COMMITTED"
assert lock["refit_protocol"]["phase"] == "DESIGN_FROZEN_BUT_REFIT_NOT_PERFORMED"

result = subprocess.run(
    [sys.executable, "scripts/verify_revised_selection_lock.py"],
    cwd=PROJECT_ROOT, capture_output=True, text=True, check=True,
)
print(result.stdout)
assert "MODEL-SELECTION LOCK PREFLIGHT: PASSED" in result.stdout
assert "NO MODEL TRAINING / NO TEST DATA ACCESS" in result.stdout
print("EXACT COMMITTED MODEL SELECTION: PASSED")
print("Checkout HEAD:", git("rev-parse", "--short", "HEAD").stdout.strip())

15 min: LightGBM F1 C1  MAE=0.177400 kW  RF challenger MAE=0.177618 kW
30 min: LightGBM F2 C1  MAE=0.219329 kW  RF challenger MAE=0.225092 kW
60 min: LightGBM F2 C1  MAE=0.280951 kW  RF challenger MAE=0.286863 kW
MODEL-SELECTION LOCK PREFLIGHT: PASSED — source and archive evidence consistent
1,665 / 1,665 candidate-fold evaluations source/identity audited
NO MODEL TRAINING / NO TEST DATA ACCESS / NO OUTPUT FILE WRITES

EXACT COMMITTED MODEL SELECTION: PASSED
Checkout HEAD: c6a9162


## 3. Reproducibility and **unfitted** estimator specification

Check the Python and four pinned model packages. Constructing an unfitted `LGBMRegressor` is allowed; calling its fitting/prediction methods is not. Match the exact `src.classical_models` frozen grid and fixed parameter policy, with no scaler, validation-dependent early stopping, or parameter changes.

In [3]:
from src.classical_models import (
    LIGHTGBM_CONFIGS, LIGHTGBM_FIXED_PARAMS, build_lightgbm_model,
)

assert sys.version_info[:2] == (3, 12), "Frozen environment requires Python 3.12"
versions = lock["reproducibility"]["dependencies_pinned_from_pyproject_toml"]
for package, expected in versions.items():
    observed = mdist.version(package)
    assert observed == expected, f"Pinned dependency mismatch: {package} {observed} != {expected}"
    print(f"{package}: {observed} (pinned)")

selections = {int(c["horizon_minutes"]): c for c in lock["selected_candidates"]}
assert set(selections) == {15, 30, 60}
assert [selections[h]["feature_family"] for h in (15, 30, 60)] == ["F1", "F2", "F2"]

spec_rows = []
for horizon in (15, 30, 60):
    spec = selections[horizon]
    config = dict(spec["configuration"])
    assert spec["model"] == "LightGBM"
    assert spec["candidate_id"] == "LightGBM_C1"
    assert spec["configuration_index_zero_based"] == 0
    assert config == dict(LIGHTGBM_CONFIGS[0])
    assert config == {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 15}
    assert LIGHTGBM_FIXED_PARAMS == lock["refit_protocol"]["lightgbm_fixed_parameters_from_frozen_source"]
    unfitted_estimator = build_lightgbm_model(config)  # constructor only — NO FIT
    actual_params = unfitted_estimator.get_params(deep=False)
    assert all(actual_params[name] == value for name, value in (LIGHTGBM_FIXED_PARAMS | config).items())
    assert not hasattr(unfitted_estimator, "booster_"), "Unexpected fitted model"
    del unfitted_estimator
    spec_rows.append({
        "horizon_minutes": horizon,
        "model": spec["model"], "feature_family": spec["feature_family"],
        "configuration": spec["candidate_id"],
        "cv_mae_kw": spec["cv_selection"]["unweighted_fold_mean_mae_kw"],
        "expected_final_refit_rows": spec["refit"]["training_rows_expected"],
    })
assert lock["refit_protocol"]["no_validation_early_stopping"] is True
specification_table = pd.DataFrame(spec_rows)
display(specification_table)
print("PINNED ENVIRONMENT AND UNFITTED MODEL CONSTRUCTION: PASSED")

lightgbm: 4.7.0 (pinned)
numpy: 2.0.2 (pinned)
pandas: 3.0.5 (pinned)
scikit-learn: 1.9.0 (pinned)


,horizon_minutes,model,feature_family,configuration,cv_mae_kw,expected_final_refit_rows
0,15,LightGBM,F1,LightGBM_C1,0.177400,104817
1,30,LightGBM,F2,LightGBM_C1,0.219329,104814
2,60,LightGBM,F2,LightGBM_C1,0.280951,104808


PINNED ENVIRONMENT AND UNFITTED MODEL CONSTRUCTION: PASSED


## 4. Timestamp-only HDF5 inspection — **no measurements loaded**

Read `DATA/axis1` and `DATA/axis0` metadata, and inspect the **shape** of `DATA/block0_values`. This does **not** read any measurement cells. Validate chronological regularity, source column availability, the exact canonical 5-minute timeline, and the exclusive pre-terminal training cutoff. We deliberately avoid the full-data loader because it would read measurements in the historically exposed test window.

In [4]:
import h5py
from src.forecasting_protocol import (
    DAILY_LAG_STEPS, F0_HISTORY_SOURCES, FORECAST_HORIZONS,
    EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON,
    MODEL_AVAILABILITY_GAP, TARGET_COLUMN, TEST_START, TRAIN_START,
    make_terminal_refit_mask,
)

H5_PATH = PROJECT_ROOT / "solete_dataset/SOLETE_Pombo_5min.h5"
if not H5_PATH.is_file():
    raise FileNotFoundError(f"Required local SOLETE 5-minute HDF5 is absent: {H5_PATH}")

CUTOFF = pd.Timestamp(lock["refit_protocol"]["development_valid_time_cutoff_utc_exclusive"])
assert CUTOFF == TEST_START - MODEL_AVAILABILITY_GAP
assert CUTOFF == pd.Timestamp("2019-05-31 23:00:00", tz="UTC")
assert tuple(FORECAST_HORIZONS) == (15, 30, 60)
assert DAILY_LAG_STEPS == 288

with h5py.File(H5_PATH, "r") as f:  # metadata only
    ds = f["DATA"]
    full_index = pd.DatetimeIndex(pd.to_datetime(ds["axis1"][:], unit="ns", utc=True), name="Timestamp")
    all_source_columns = tuple(x.decode("utf-8") if isinstance(x, bytes) else str(x)
                               for x in ds["axis0"][:])
    measurement_shape = ds["block0_values"].shape

assert measurement_shape == (len(full_index), len(all_source_columns))
assert len(full_index) == 131617
assert full_index[0] == TRAIN_START
assert full_index[-1] == pd.Timestamp("2019-09-01 00:00:00", tz="UTC")
assert full_index.is_monotonic_increasing and full_index.is_unique
assert (full_index[1:] - full_index[:-1] == pd.Timedelta(minutes=5)).all()
required_raw = (set(F0_HISTORY_SOURCES.values()) - {"Humidity_clean", "Pressure_clean[mbar]",
                  "wind_direction_sin", "wind_direction_cos"}) | {
    TARGET_COLUMN, "Pressure[mbar]", "HUMIDITY[%]", "WIND_DIR[deg]"
}
assert required_raw.issubset(all_source_columns), f"HDF5 columns missing: {sorted(required_raw-set(all_source_columns))}"

n_development = int(full_index.searchsorted(CUTOFF, side="left"))
assert 0 < n_development < len(full_index)
assert full_index[n_development - 1] < CUTOFF <= full_index[n_development]
assert n_development == 105108
print("HDF5 timestamp/schema metadata: PASSED")
print("Available development-only source rows:", n_development)
print("Last admissible measurement timestamp:", full_index[n_development-1])
print("Held-out measurement cells read: ZERO")

HDF5 timestamp/schema metadata: PASSED
Available development-only source rows: 105108
Last admissible measurement timestamp: 2019-05-31 22:55:00+00:00
Held-out measurement cells read: ZERO


## 5. Independently reconstruct the three final-refit masks **from timestamp metadata alone**

These issue/valid-time identities require no measured target values. The first permissible issue must have a complete 24-hour lag; the latest issue must permit its target to arrive before the fixed 60-minute embargo. Compare independently computed populations against the frozen protocol's `make_terminal_refit_mask` and archive row counts.

In [5]:
issue_time_by_horizon = {}
geometry = []
for horizon, horizon_steps in FORECAST_HORIZONS.items():
    expected_rows = int(EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON[horizon])
    expected_locked = int(selections[horizon]["refit"]["training_rows_expected"])
    assert expected_rows == expected_locked
    issues = full_index[DAILY_LAG_STEPS:n_development-horizon_steps]
    valid = pd.Series(issues + pd.Timedelta(minutes=horizon), index=issues, name="valid_time")
    mask = make_terminal_refit_mask(valid)
    assert mask.all() and int(mask.sum()) == expected_rows
    assert len(issues) == expected_rows
    assert issues[0] == pd.Timestamp("2018-06-02 00:00:00", tz="UTC")
    assert valid.iloc[-1] == CUTOFF - pd.Timedelta(minutes=5)
    assert valid.iloc[-1] < CUTOFF < TEST_START
    assert valid.iloc[-1] < TEST_START
    assert (pd.DatetimeIndex(valid.array) > issues).all()
    issue_time_by_horizon[horizon] = issues
    geometry.append({
        "horizon_minutes": horizon, "eligible_training_rows": len(issues),
        "first_issue_utc": str(issues[0]), "last_issue_utc": str(issues[-1]),
        "last_training_label_valid_utc": str(valid.iloc[-1]),
    })
geometry_table = pd.DataFrame(geometry)
display(geometry_table)
assert sum(x["eligible_training_rows"] for x in geometry) == 314439
print("FINAL-REFIT TIMELINE PREFLIGHT: PASSED — 104817 / 104814 / 104808")
print("No terminal measurements, future targets, predictions or fits read/performed")

,horizon_minutes,eligible_training_rows,first_issue_utc,last_issue_utc,last_training_label_valid_utc
0,15,104817,2018-06-02 00:00:00+00:00,2019-05-31 22:40:00+00:00,2019-05-31 22:55:00+00:00
1,30,104814,2018-06-02 00:00:00+00:00,2019-05-31 22:25:00+00:00,2019-05-31 22:55:00+00:00
2,60,104808,2018-06-02 00:00:00+00:00,2019-05-31 21:55:00+00:00,2019-05-31 22:55:00+00:00


FINAL-REFIT TIMELINE PREFLIGHT: PASSED — 104817 / 104814 / 104808
No terminal measurements, future targets, predictions or fits read/performed


## 6. Development-only feature/target preflight (no training)

This reads **only HDF5 measurement rows `0:n_development`**, never later rows, then applies the established causal cleaning and frozen F0 construction. For each selected horizon, F1 Gregorian and (where selected) F2 solar features are calculated at the **forecast-valid timestamp**, using the existing +2-minute astronomical convention for F2. No F3 or F4 features are constructed here. This is an input-only parity and leakage check; **no model is fitted, predictions made, or scores computed**.

This is the relatively expensive part of the notebook. It runs by default for full verification. Setting `RUN_DEVELOPMENT_INPUT_CHECK=False` will skip it, in which case the notebook must report **INCOMPLETE**, not PASSED.

In [6]:
RUN_DEVELOPMENT_INPUT_CHECK = True  # SAFE read-only check of PRE-CUTOFF measurement values ONLY
REFIT_AUTHORIZED = False
TERMINAL_EVALUATION_AUTHORIZED = False
assert not REFIT_AUTHORIZED and not TERMINAL_EVALUATION_AUTHORIZED
assert RUN_DEVELOPMENT_INPUT_CHECK is True, (
    "Full Notebook 13 acceptance requires the development-only input check"
)

# An explicit HDF5 hyperslab prevents reading ANY held-out measurement/target value.
with h5py.File(H5_PATH, "r") as f:
    source_values_development = f["DATA/block0_values"][:n_development, :]
assert source_values_development.shape == (n_development, len(all_source_columns))
raw_development = pd.DataFrame(
    source_values_development, index=full_index[:n_development], columns=all_source_columns
)
del source_values_development
assert raw_development.index[-1] < CUTOFF
assert not (raw_development.index >= CUTOFF).any()
print("Development measurement rows loaded:", len(raw_development))
print("Latest loaded observation:", raw_development.index[-1])
print("Terminal measurement/target rows loaded: ZERO")

Development measurement rows loaded: 105108
Latest loaded observation: 2019-05-31 22:55:00+00:00
Terminal measurement/target rows loaded: ZERO


In [7]:
from src.preprocessing import build_cleaned_solete_features, build_gregorian_features
from src.forecasting_protocol import build_f0_history, build_forecast_targets
from src.astronomy_features import (
    SOLETE_ASTRONOMY_OFFSET, SOLETE_SOLAR_ALTITUDE_CONVENTION_M,
    build_solar_f2_features,
)

assert SOLETE_ASTRONOMY_OFFSET == pd.Timedelta(minutes=2)
assert SOLETE_SOLAR_ALTITUDE_CONVENTION_M == 0.0
clean_dev = build_cleaned_solete_features(raw_development)
f0_dev = build_f0_history(clean_dev)
targets_dev = build_forecast_targets(clean_dev)
assert len(f0_dev.columns) == 118

input_audit_rows = []
for horizon in (15, 30, 60):
    spec = selections[horizon]
    label = targets_dev[horizon]
    eligible = (
        f0_dev.notna().all(axis=1)
        & label["target_P_Solar_kW"].notna()
        & label["valid_time"].lt(CUTOFF)
    )
    issue_idx = clean_dev.index[eligible]
    assert issue_idx.equals(issue_time_by_horizon[horizon]), (
        f"{horizon} min: development issue-time index mismatch"
    )
    valid_idx = pd.DatetimeIndex(label.loc[issue_idx, "valid_time"], name="valid_time")
    y = label.loc[issue_idx, "target_P_Solar_kW"]
    assert valid_idx.max() < CUTOFF
    assert valid_idx.isin(raw_development.index).all()
    assert all((issue_idx + pd.Timedelta(minutes=horizon)) == valid_idx)

    f0 = f0_dev.loc[issue_idx].rename(columns={"P_Solar_kw_lag_288": "P_Solar_kw_issue_minus_24h"})
    f1_increment = build_gregorian_features(valid_idx).set_axis(issue_idx).add_prefix("valid_")
    X = pd.concat([f0, f1_increment], axis=1)
    if spec["feature_family"] == "F2":
        f2_increment = build_solar_f2_features(valid_idx).set_axis(issue_idx).add_prefix("valid_")
        assert f2_increment.shape[1] == 5
        X = pd.concat([X, f2_increment], axis=1)
    else:
        assert spec["feature_family"] == "F1"
    assert X.index.equals(y.index) and X.columns.is_unique
    assert len(X) == spec["refit"]["training_rows_expected"]
    expected_features = {"F1": 133, "F2": 138}[spec["feature_family"]]
    assert X.shape == (len(issue_idx), expected_features)
    assert not X.isna().any().any() and np.isfinite(X.to_numpy(dtype=float)).all()
    assert not y.isna().any() and np.isfinite(y.to_numpy(dtype=float)).all()
    assert list(X.columns[:118]) == list(f0.columns)
    assert list(X.columns[118:133]) == list(f1_increment.columns)
    # Feature order is identical to Notebook 10: F0 -> valid-time F1 -> valid-time F2.
    schema_sha256 = hashlib.sha256(json.dumps(list(X.columns), ensure_ascii=False).encode()).hexdigest()
    input_audit_rows.append({
        "horizon_minutes": horizon, "feature_family": spec["feature_family"],
        "training_rows": len(issue_idx), "predictor_columns": X.shape[1],
        "last_label_utc": str(valid_idx.max()),
        "ordered_column_names_sha256": schema_sha256,
        "finite_features_and_target": True,
    })
    del X, y, f0, f1_increment
    if "f2_increment" in locals():
        del f2_increment

input_preflight_summary = pd.DataFrame(input_audit_rows)
assert len(input_preflight_summary) == 3
assert input_preflight_summary["training_rows"].tolist() == [104817, 104814, 104808]
assert input_preflight_summary["predictor_columns"].tolist() == [133, 138, 138]
display(input_preflight_summary)
# No matrices are saved, and no estimator.fit()/predict() is called.
del raw_development, clean_dev, f0_dev, targets_dev
print("DEVELOPMENT-ONLY F1/F2 INPUT PREFLIGHT: PASSED")
print("Terminal measurements/labels read: ZERO")

,horizon_minutes,feature_family,training_rows,predictor_columns,last_label_utc,ordered_column_names_sha256,finite_features_and_target
0,15,F1,104817,133,2019-05-31 22:55:00+00:00,abacbd22f6a80e771a56ee5a0c18d74e860a5775c990e4...,True
1,30,F2,104814,138,2019-05-31 22:55:00+00:00,6cfaaebb47ace957a7d016d8e6fd733b391bda2b0ccf82...,True
2,60,F2,104808,138,2019-05-31 22:55:00+00:00,6cfaaebb47ace957a7d016d8e6fd733b391bda2b0ccf82...,True


DEVELOPMENT-ONLY F1/F2 INPUT PREFLIGHT: PASSED
Terminal measurements/labels read: ZERO


## 7. Final acceptance report — **no refit authority**

Accept only when the complete source/archive audit, pinned environment, mask reconstruction and **real development-only** input construction all passed in the same execution. Do not transfer a model-selection decision from the historical Notebooks 12–15; do not change the three frozen candidates based on future evidence. The older SOLETE June–August 2019 window is **historically exposed**, so a future retrospective report must label it accordingly. A genuinely independent evaluation requires new, protected data and separately approved protocol.

In [8]:
assert not REFIT_AUTHORIZED and not TERMINAL_EVALUATION_AUTHORIZED
assert len(specification_table) == len(geometry_table) == len(input_preflight_summary) == 3
assert input_preflight_summary["training_rows"].tolist() == geometry_table["eligible_training_rows"].tolist()
assert input_preflight_summary["feature_family"].tolist() == ["F1", "F2", "F2"]
print("NOTEBOOK 13 FINAL-REFIT PREFLIGHT: PASSED")
print("Selected LightGBM C1 candidates: F1 / F2 / F2")
print("Development-only refit rows: 104817 / 104814 / 104808")
print("Committed lock SHA verification and 1665-candidate-fold audit: PASSED")
print("Training fits: ZERO | Predictions: ZERO | Terminal labels accessed: ZERO")
print("MODEL REFIT: NOT AUTHORIZED | TERMINAL EVALUATION: NOT AUTHORIZED")

NOTEBOOK 13 FINAL-REFIT PREFLIGHT: PASSED
Selected LightGBM C1 candidates: F1 / F2 / F2
Development-only refit rows: 104817 / 104814 / 104808
Committed lock SHA verification and 1665-candidate-fold audit: PASSED
Training fits: ZERO | Predictions: ZERO | Terminal labels accessed: ZERO
MODEL REFIT: NOT AUTHORIZED | TERMINAL EVALUATION: NOT AUTHORIZED


## Next decision (not executed here)

When this notebook passes on your local machine, save and commit its executed, read-only audit outputs separately. **Do not** edit the frozen `src/*.py` files, prior notebooks or the committed selection manifest. A future **separate, explicitly authorized** refit runner could consume these exact F1/F2 inputs, refit **one locked LightGBM per horizon** using all admissible development labels, and save model provenance without looking at terminal outcomes. Refit and any historically exposed terminal evaluation require distinct decisions. Separate F3→F4 Panchang mechanism studies should not be conflated with this primary forecast-model selection.